In [48]:
import pandas as pd
import re
import joblib

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix



In [ ]:
df = pd.read_csv("../dataset/go_emotions_dataset.csv")

print("Dataset shape:", df.shape)
print(df.head())

emotion_columns = [
    c for c in df.columns
    if c not in ["id", "text", "example_very_unclear"]
]

df = df[df[emotion_columns].sum(axis=1) > 0].copy()
df = df[df[emotion_columns].sum(axis=1) == 1].copy()

positive_tags = [
    "admiration", "amusement", "approval", "caring", "curiosity", "desire",
    "excitement", "gratitude", "joy", "love", "optimism", "pride", "relief"
]
negative_tags = [
    "anger", "annoyance", "disappointment", "disapproval", "disgust",
    "embarrassment", "fear", "grief", "nervousness", "remorse", "sadness"
]

df = df[df[positive_tags + negative_tags].sum(axis=1) == 1].copy()

df["label"] = df.apply(lambda row: "positive" if row[positive_tags].sum() == 1 else "negative", axis=1)

print("Label distribution:")
print(df["label"].value_counts())

text_column = "text"
label_column = "label"

Dataset shape: (211225, 31)
        id                                               text  \
0  eew5j0j                                    That game hurt.   
1  eemcysk   >sexuality shouldn’t be a grouping category I...   
2  ed2mah1     You do right, if you don't care then fuck 'em!   
3  eeibobj                                 Man I love reddit.   
4  eda6yn6  [NAME] was nowhere near them, he was by the Fa...   

   example_very_unclear  admiration  amusement  anger  annoyance  approval  \
0                 False           0          0      0          0         0   
1                  True           0          0      0          0         0   
2                 False           0          0      0          0         0   
3                 False           0          0      0          0         0   
4                 False           0          0      0          0         0   

   caring  confusion  ...  love  nervousness  optimism  pride  realization  \
0       0          0  ...     0   

In [ ]:
df = df.dropna(subset=[text_column, label_column])

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+", "", text)          # remove links
    text = re.sub(r"@\w+", "", text)             # remove mentions
    text = re.sub(r"#\w+", "", text)             # remove hashtags
    text = re.sub(r"[^a-zA-Z\s]", "", text)      # remove numbers/symbols
    text = re.sub(r"\s+", " ", text)             # remove extra spaces
    return text.strip()

df["clean_text"] = df[text_column].apply(clean_text)


In [ ]:
X = df["clean_text"]
y = df[label_column]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
model = Pipeline([
    ("tfidf", TfidfVectorizer(
        analyzer="char_wb",
        ngram_range=(3, 6),
        max_features=40000,
        sublinear_tf=True
    )),
    ("classifier", LinearSVC(
        class_weight="balanced",
        max_iter=5000
    ))

])

In [ ]:
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Classification Report:")
print(classification_report(y_test, y_pred))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.8012088974854932
Classification Report:
              precision    recall  f1-score   support

    negative       0.70      0.81      0.75      7709
    positive       0.88      0.79      0.83     12971

    accuracy                           0.80     20680
   macro avg       0.79      0.80      0.79     20680
weighted avg       0.81      0.80      0.80     20680

Confusion Matrix:
[[ 6264  1445]
 [ 2666 10305]]


In [ ]:
texts = [
    "I feel very sad and lonely today",
    "I am excited and happy about my exam results",
    "I am scared about my future",
    "This situation makes me angry"
]

predictions = model.predict(texts)

for text, emotion in zip(texts, predictions):
    print(text, "=>", emotion)

I feel very sad and lonely today => negative
I am excited and happy about my exam results => positive
I am scared about my future => negative
This situation makes me angry => negative


In [ ]:
import joblib

joblib.dump(model,"../models/emotion_model.pkl")

['../models/emotion_model.pkl']